<a href="https://colab.research.google.com/github/ust-buan-640/buan-640-notebooks/blob/main/notebooks/deming-red-bead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Red Bead Factory

This notebook is based on W. Edwards Deming's Red Bead Experiment, which he ran in his management seminars for decades. We will play the game first and do the numbers afterward.

**How to use this notebook.** Click a gray code cell and press **Shift + Enter** to run it, or press the ▶ button on its left. Run the cells in order, from top to bottom. If something breaks, go to **Runtime → Restart session** and run everything again from the top.

**During class**, the instructor runs Sections 1–3 on the projector. You join in at **Section 4**, but you still need to run the setup cell in Section 1 first.

## 1. Setup

This cell loads the Python packages we need and builds the bead box. Run it once at the start.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, HTML, JSON

rng = np.random.default_rng()

# The factory
BOX_SIZE = 4000      # beads in the box
RED_IN_BOX = 800     # red beads (20%)
PADDLE = 50          # wells in the paddle (5 rows x 10 columns)

box = np.array([1] * RED_IN_BOX + [0] * (BOX_SIZE - RED_IN_BOX))   # 1 = red, 0 = white

def dip_paddle(red_weight=1.0):
    """Dip the paddle into the box and return 50 beads (1 = red, 0 = white).
    red_weight < 1 makes red beads less likely to settle into a well."""
    w = np.where(box == 1, red_weight, 1.0)
    idx = rng.choice(BOX_SIZE, size=PADDLE, replace=False, p=w / w.sum())
    return box[idx]

# Game state (empty until the instructor plays the game)
state = {"workers": [], "schedule": [], "pos": 0, "log": []}

try:
    from google.colab import output as colab_output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Setup complete. Running in Colab:", IN_COLAB)

Setup complete. Running in Colab: True


## 2. The factory

Our company produces white beads. Customers do not want red beads. The raw material arrives in a box of 4,000 beads, and 800 of them, or 20%, are red. Here is the box after a thorough mixing.

In [ ]:
mixed = rng.permutation(box)
xs = rng.uniform(0, 1, BOX_SIZE)
ys = rng.uniform(0, 1, BOX_SIZE)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(xs[mixed == 0], ys[mixed == 0], s=9, c="#f4f1ea", edgecolors="#b8b2a6", linewidths=0.3)
ax.scatter(xs[mixed == 1], ys[mixed == 1], s=9, c="#c0392b")
ax.set_xticks([]); ax.set_yticks([])
ax.set_facecolor("#7a5230")
ax.set_title(f"The bead box: {BOX_SIZE:,} beads, {RED_IN_BOX} of them red ({RED_IN_BOX / BOX_SIZE:.0%})")
plt.show()

## 3. Production (instructor on the projector)

### 3a. Hire the workers

Type the names of the six willing workers in the order they will work. If you rerun this cell, the game starts over.

In [ ]:
WORKERS = ["Worker 1", "Worker 2", "Worker 3", "Worker 4", "Worker 5", "Worker 6"]

def reset_game(names):
    state["workers"] = list(names)
    state["schedule"] = [(day, w) for day in (1, 2, 3) for w in names]
    state["pos"] = 0
    state["log"] = []

reset_game(WORKERS)
print("Hired:", ", ".join(WORKERS))
print(f"Production plan: 3 days, {len(state['schedule'])} paddle dips.")

### 3b. The Standard Operating Procedure

Read this before Day 1. Every worker must follow it exactly.

1. Wait for the foreman to say **"Go."**
2. Press the clicker **once**, with the **right index finger**.
3. Do **not** look at the screen while pressing.
4. Hand the clicker to the next worker without comment.

The two inspectors count the red beads aloud from the screen, independently. The chief inspector announces the official count, and the recorder writes it on the whiteboard.

### 3c. The production line

Run the cell below, then **click once on the paddle panel** so it has the keyboard focus. After that, the clicker (Page Down, arrow keys, or the spacebar) dips the paddle. You can also use the button.

If the clicker stops working, someone clicked outside the panel. Click on the panel again.

In [ ]:
import random

GOOD = [
    "Outstanding, {w}! Only {r} red beads. This is what commitment looks like.",
    "{r}! Everyone, look at {w}. That is how a professional presses a clicker.",
    "Magnificent work, {w}. {r} red beads. You are going places in this company.",
    "{w}, {r} red beads. I want everyone to watch {w}'s technique very carefully.",
]
OK = [
    "{r}. Acceptable, {w}, but acceptable is not excellence.",
    "{r} red beads. I expected more from you, {w}. Focus.",
    "Hmm. {r}. Not bad, not good. Management is watching, {w}.",
    "{r}. Average, {w}. This company was not built on average.",
]
BAD = [
    "{r}?! {w}, are you even trying? {best} got {best_r}. Study {best}'s technique.",
    "{r} red beads. {w}, the customers will see these. Think about the customers.",
    "{r}. I don't know what to say, {w}. Did you read the Standard Operating Procedure?",
    "{r}! {w}, this is exactly the kind of attitude that is hurting this company.",
]
BAD_NO_BEST = [
    "{r}?! {w}, are you even trying?",
    "{r} red beads. {w}, the customers will see these. Think about the customers.",
]

def foreman_comment(worker, red):
    others = [e for e in state["log"] if e["worker"] != worker]
    if red <= 7:
        return random.choice(GOOD).format(w=worker, r=red)
    if red <= 12:
        return random.choice(OK).format(w=worker, r=red)
    if others:
        best = min(others, key=lambda e: e["red"])
        if best["red"] < red:
            return random.choice(BAD).format(w=worker, r=red, best=best["worker"], best_r=best["red"])
    return random.choice(BAD_NO_BEST).format(w=worker, r=red)

def next_up():
    if state["pos"] >= len(state["schedule"]):
        return None
    day, w = state["schedule"][state["pos"]]
    return {"day": day, "worker": w}

def production_step():
    """One paddle dip by the next worker in the schedule. Returns a dict."""
    nxt = next_up()
    if nxt is None:
        return {"done": True}
    beads = dip_paddle()
    red = int(beads.sum())
    comment = foreman_comment(nxt["worker"], red)
    state["log"].append({"draw": len(state["log"]) + 1, "day": nxt["day"],
                         "worker": nxt["worker"], "red": red})
    state["pos"] += 1
    after = next_up()
    day_done = after is None or after["day"] != nxt["day"]
    return {"done": False, "day": nxt["day"], "worker": nxt["worker"],
            "beads": beads.tolist(), "red": red, "comment": comment,
            "day_done": day_done, "next": after}

def undo_last():
    """Remove the last dip, e.g. after a misclick."""
    if state["log"]:
        removed = state["log"].pop()
        state["pos"] -= 1
        print("Removed:", removed)

PANEL = r"""
<div id="deming-root" tabindex="0" style="outline:none; font-family:Arial, sans-serif; max-width:820px;
     background:#faf7f0; border:3px solid #7a5230; border-radius:12px; padding:18px;">
  <div id="d-hdr" style="font-size:26px; font-weight:bold; color:#4a2f18;">__HEADER__</div>
  <div id="d-sub" style="font-size:15px; color:#6b5a45; margin-bottom:10px;">Click once on this panel, then use the clicker.</div>
  <svg id="d-paddle" width="780" height="330" viewBox="0 0 780 330"></svg>
  <div id="d-count" style="font-size:44px; font-weight:bold; height:56px; margin-top:6px;"></div>
  <div id="d-comment" style="font-size:22px; font-style:italic; color:#333; min-height:64px;"></div>
  <div id="d-next" style="font-size:16px; color:#6b5a45; margin-top:6px;">__NEXT__</div>
  <button id="d-btn" style="margin-top:12px; font-size:18px; padding:8px 22px; background:#7a5230;
          color:white; border:none; border-radius:6px; cursor:pointer;">Dip the paddle</button>
</div>
<script>
(function () {
  const root = document.getElementById("deming-root");
  const svg = document.getElementById("d-paddle");
  const hdr = document.getElementById("d-hdr");
  const sub = document.getElementById("d-sub");
  const cnt = document.getElementById("d-count");
  const com = document.getElementById("d-comment");
  const nxt = document.getElementById("d-next");
  const NS = "http://www.w3.org/2000/svg";
  const KEYS = new Set(["PageDown", "PageUp", "ArrowRight", "ArrowLeft", "ArrowDown",
                        "ArrowUp", " ", "Enter", "b", "B", "."]);
  const ROW_DELAY = 550;
  let busy = false;

  function drawPaddle(beads, rowsShown) {
    while (svg.firstChild) svg.removeChild(svg.firstChild);
    const board = document.createElementNS(NS, "rect");
    board.setAttribute("x", 10); board.setAttribute("y", 10);
    board.setAttribute("width", 760); board.setAttribute("height", 310);
    board.setAttribute("rx", 18); board.setAttribute("fill", "#a0703f");
    svg.appendChild(board);
    for (let r = 0; r < 5; r++) {
      for (let c = 0; c < 10; c++) {
        const i = r * 10 + c;
        const circ = document.createElementNS(NS, "circle");
        circ.setAttribute("cx", 65 + c * 72); circ.setAttribute("cy", 45 + r * 60);
        circ.setAttribute("r", 24);
        let fill = "#5c3d1e", stroke = "#3d2812";
        if (beads && r < rowsShown) {
          fill = beads[i] === 1 ? "#c0392b" : "#f4f1ea";
          stroke = beads[i] === 1 ? "#7b241c" : "#b8b2a6";
        }
        circ.setAttribute("fill", fill); circ.setAttribute("stroke", stroke);
        circ.setAttribute("stroke-width", 2);
        svg.appendChild(circ);
      }
    }
  }

  function nextText(n) {
    return n ? "Next up: " + n.worker + " (Day " + n.day + ")" : "Production plan complete.";
  }

  async function dip() {
    if (busy) return;
    busy = true;
    cnt.textContent = ""; com.textContent = "";
    try {
      const res = await google.colab.kernel.invokeFunction("deming.dip", [], {});
      const d = res.data["application/json"];
      if (d.done) {
        hdr.textContent = "Production plan complete";
        com.textContent = "Run the bulletin cell below.";
        busy = false;
        return;
      }
      hdr.textContent = "Day " + d.day + ": " + d.worker;
      sub.textContent = "Dipping...";
      drawPaddle(d.beads, 0);
      for (let r = 1; r <= 5; r++) {
        setTimeout(() => drawPaddle(d.beads, r), r * ROW_DELAY);
      }
      setTimeout(() => {
        sub.textContent = "Inspectors, count the red beads.";
        cnt.textContent = d.red + " red";
        cnt.style.color = d.red <= 7 ? "#1e7b34" : (d.red <= 12 ? "#7a5230" : "#c0392b");
        com.textContent = "\u201C" + d.comment + "\u201D";
        nxt.textContent = (d.day_done ? "End of Day " + d.day + ". Run the bulletin cell. " : "") + nextText(d.next);
        busy = false;
        root.focus();
      }, 6 * ROW_DELAY + 300);
    } catch (e) {
      com.textContent = "Error: " + e + ". Rerun this cell and try again.";
      busy = false;
    }
  }

  document.addEventListener("keydown", (e) => {
    if (KEYS.has(e.key)) { e.preventDefault(); dip(); }
  });
  document.getElementById("d-btn").onclick = dip;
  drawPaddle(null, 0);
  root.focus();
})();
</script>
"""

def _dip_callback():
    return JSON(production_step())

if IN_COLAB:
    colab_output.register_callback("deming.dip", _dip_callback)
    n = next_up()
    header = "The production line is open" if n else "Production plan complete"
    nxt_text = f"Next up: {n['worker']} (Day {n['day']})" if n else "Run the bulletin cell."
    display(HTML(PANEL.replace("__HEADER__", header).replace("__NEXT__", nxt_text)))
else:
    print("The clicker panel only works in Google Colab.")
    print("Outside Colab, call production_step() once per dip, for example:")
    print("    production_step()")

### 3d. The management bulletin

Run this cell at the end of each day.

In [ ]:
def announce_firings():
    """After Day 3: fire the three workers with the most red beads; the best three work a double shift on Day 4."""
    df = pd.DataFrame(state["log"])
    df = df[df["day"] <= 3]
    totals = df.groupby("worker")["red"].sum().sort_values(kind="stable")
    best3 = list(totals.index[:3])
    worst3 = list(totals.index[-3:])
    state["best3"], state["fired"] = best3, worst3
    day4 = [(4, w) for w in best3] + [(4, w) for w in best3]
    state["schedule"] = [s for s in state["schedule"] if s[0] <= 3] + day4
    return best3, worst3, totals

def daily_totals_html(df):
    totals = df.groupby("day")["red"].sum()
    dips = df.groupby("day")["red"].size()
    cells_day = "".join(f"<th style='padding:2px 12px'>Day {d}</th>" for d in totals.index)
    cells_tot = "".join(f"<td style='padding:2px 12px; text-align:center; font-size:20px'><b>{t}</b></td>"
                        for t in totals.values)
    cells_dips = "".join(f"<td style='padding:2px 12px; text-align:center; color:#777'>{n} dips</td>"
                         for n in dips.values)
    return ("<b>Total red beads per day</b>"
            "<table style='margin:6px 0 12px 0; border-collapse:collapse'>"
            f"<tr>{cells_day}</tr><tr>{cells_tot}</tr><tr>{cells_dips}</tr></table>")

def bulletin():
    if not state["log"]:
        print("No production yet.")
        return
    day = state["log"][-1]["day"]
    df = pd.DataFrame(state["log"])
    today = df[df["day"] == day].sort_values("red", kind="stable")
    best = today.iloc[0]
    worst = today.iloc[-1]

    box_style = ("font-family:Arial, sans-serif; max-width:760px; background:#fffdf5; border:2px solid #333;"
                 "padding:16px 22px; font-size:17px;")
    parts = [f"<div style='{box_style}'>",
             f"<div style='font-size:24px; font-weight:bold;'>MANAGEMENT BULLETIN: DAY {day}</div><hr>"]

    rows = "".join(f"<tr><td style='padding:2px 14px 2px 0'>{r.worker}</td>"
                   f"<td style='text-align:right'>{r.red}</td></tr>" for r in today.itertuples())
    parts.append("<b>Today's results</b><table style='margin:6px 0 12px 0'>"
                 "<tr><th style='text-align:left; padding-right:14px'>Worker</th><th>Red beads</th></tr>"
                 f"{rows}</table>")
    parts.append(daily_totals_html(df))

    if day == 1:
        parts.append(f"<p>&#127942; <b>Employee of the Day: {best.worker}</b>, with only {best.red} red beads. "
                     "Please collect your reward from the foreman.</p>")
        parts.append(f"<p>&#9888;&#65039; <b>Written warning: {worst.worker}</b> ({worst.red} red beads). "
                     "Management expects immediate improvement.</p>")
        parts.append("<p>Our target is <b>no more than 3 red beads per paddle</b>. Every worker can meet "
                     "this target if they follow the procedure and try harder.</p>")
    elif day == 2:
        parts.append("<p style='font-size:26px; font-weight:bold; text-align:center; color:#c0392b; "
                     "border:3px dashed #c0392b; padding:8px;'>QUALITY IS JOB ONE!<br>DO IT RIGHT THE FIRST TIME!</p>")
        parts.append(f"<p>&#127942; <b>Merit bonus: {best.worker}</b> ({best.red} red beads).</p>")
        on_prob = list(today.loc[today["red"] >= 12, "worker"])
        if worst.worker not in on_prob:
            on_prob.append(worst.worker)
        parts.append("<p>&#9888;&#65039; <b>On probation:</b> " + ", ".join(on_prob) + ".</p>")
        parts.append("<p>Management notes that the red bead count has <b>not</b> met the target. "
                     "Tomorrow is your last chance.</p>")
    elif day == 3:
        best3, worst3, totals = announce_firings()
        rows3 = "".join(f"<tr><td style='padding:2px 14px 2px 0'>{w}</td>"
                        f"<td style='text-align:right'>{t}</td></tr>" for w, t in totals.items())
        parts.append("<b>Total red beads per worker, Days 1&ndash;3</b><table style='margin:6px 0 12px 0'>"
                     f"{rows3}</table>")
        parts.append("<p>Management has reviewed three days of performance and is taking decisive action.</p>")
        parts.append("<p>&#10060; <b>Dismissed, effective immediately:</b> " + ", ".join(worst3) + ".</p>")
        parts.append("<p>&#11088; <b>Our three best workers</b>, " + ", ".join(best3) +
                     ", will work a <b>double shift</b> on Day 4 to cover. Surely the results will now improve.</p>")
        parts.append("<p><i>Run the production line cell again to start Day 4.</i></p>")
    else:
        parts.append("<p>Day 4 was produced entirely by our three best workers. "
                     "Compare today's total with the previous days.</p>")
        parts.append("<p><i>Run the Day 4 review below.</i></p>")
    parts.append("</div>")
    display(HTML("".join(parts)))

bulletin()

### 3e. Day 4 review

Run this after the double shift. Did the best workers stay the best?

In [ ]:
def day4_review():
    if "best3" not in state:
        print("Run the Day 3 bulletin first.")
        return
    df = pd.DataFrame(state["log"])
    before = df[(df["day"] <= 3) & (df["worker"].isin(state["best3"]))].groupby("worker")["red"].mean()
    after = df[df["day"] == 4].groupby("worker")["red"].mean()
    out = pd.DataFrame({"Average, Days 1-3": before.round(1), "Average, Day 4": after.round(1)})
    display(out)
    print(f"Average for the 'best' workers, Days 1-3: {before.mean():.1f} red beads")
    print(f"Average for the same workers, Day 4:     {after.mean():.1f} red beads")
    print(f"Average for all workers, Days 1-3:       {df[df['day'] <= 3]['red'].mean():.1f} red beads")

day4_review()

## 4. What actually happened? (everyone)

From here on, everyone works in their own copy of the notebook. Make sure you ran the setup cell in Section 1.

Type tonight's counts from the whiteboard into the list below, in order, separated by commas. If you are the instructor and played the game in this notebook, you can leave the list empty and the counts are taken from the game automatically.

In [ ]:
tonight = []   # for example: tonight = [9, 12, 7, 10, 14, 8, ...]

if tonight:
    counts = np.array(tonight)
elif state["log"]:
    counts = np.array([e["red"] for e in state["log"]])
else:
    counts = rng.hypergeometric(RED_IN_BOX, BOX_SIZE - RED_IN_BOX, PADDLE, size=24)
    print("No class data entered, so these are 24 simulated dips.")

print("Red beads per paddle:", counts.tolist())
print("Number of dips:", len(counts), "| Total red beads:", counts.sum())

### 4a. Control limits, by hand first

Before running the next cell, compute these on paper. Each paddle is a sample of $n = 50$ beads.

1. The average proportion of red beads over all dips: $\bar p = \dfrac{\text{total red beads}}{\text{number of dips} \times 50}$
2. The standard deviation of the sample proportion: $\sigma_{\bar p} = \sqrt{\dfrac{\bar p\,(1-\bar p)}{n}}$
3. The three-sigma control limits: $UCL = \bar p + 3\sigma_{\bar p}$ and $LCL = \bar p - 3\sigma_{\bar p}$

Multiply the limits by 50 to translate them into numbers of red beads. Then run the cell and check your work.

In [ ]:
n = PADDLE
p_hat = counts / n
p_bar = counts.sum() / (len(counts) * n)
sigma = np.sqrt(p_bar * (1 - p_bar) / n)
UCL = p_bar + 3 * sigma
LCL = max(0, p_bar - 3 * sigma)

print(f"p-bar = {p_bar:.3f}")
print(f"sigma = {sigma:.4f}")
print(f"UCL   = {UCL:.3f}  (about {UCL * n:.1f} red beads)")
print(f"LCL   = {LCL:.3f}  (about {LCL * n:.1f} red beads)")

### 4b. The p-chart

In [ ]:
x = np.arange(1, len(counts) + 1)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(x, p_hat, marker="o", color="#2c3e50", zorder=3)
ax.axhline(p_bar, color="gray", label=f"center line = {p_bar:.3f}")
ax.axhline(UCL, color="#c0392b", linestyle="--", label=f"UCL = {UCL:.3f}")
ax.axhline(LCL, color="#c0392b", linestyle="--", label=f"LCL = {LCL:.3f}")

# shade the production days, if we know them
if not tonight and state["log"]:
    days = [e["day"] for e in state["log"]]
    for d in sorted(set(days)):
        idx = [i + 1 for i, dd in enumerate(days) if dd == d]
        if d % 2 == 0:
            ax.axvspan(min(idx) - 0.5, max(idx) + 0.5, color="#f0e6d2", zorder=0)
        ax.text(np.mean(idx), UCL + 0.02, f"Day {d}", ha="center", fontsize=10)

ax.set_xlabel("Paddle dip")
ax.set_ylabel("Proportion of red beads")
ax.set_title("p-chart for the Red Bead Factory")
ax.set_ylim(0, max(UCL + 0.08, p_hat.max() + 0.05))
ax.legend(loc="lower right")
plt.show()

out = x[(p_hat > UCL) | (p_hat < LCL)]
if len(out):
    print("Points outside the control limits: dips", out.tolist())
else:
    print("Every point is inside the control limits.")

**Discuss with your neighbor.**

- Is this process in statistical control?
- Was any worker really better or worse than the others? What does the chart say about the Employee of the Day and the workers who were fired?
- The foreman's target was at most 3 red beads per paddle. Where is that on the chart? What would it take to meet it?

## 5. Ten thousand days of production

We only had time for a handful of dips. The computer can run the factory for 10,000 days. Each dip takes 50 beads from a box with 800 red and 3,200 white beads, so the number of red beads in a paddle follows (almost exactly) a binomial distribution with $n = 50$ and $p = 0.2$, which you met in Module 3.

In [ ]:
sim = rng.hypergeometric(RED_IN_BOX, BOX_SIZE - RED_IN_BOX, PADDLE, size=10_000)

k = np.arange(0, 26)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(sim, bins=np.arange(-0.5, 26.5, 1), density=True, color="#e8dcc6", edgecolor="#7a5230",
        label="10,000 simulated dips")
ax.plot(k, stats.binom.pmf(k, PADDLE, 0.2), "o-", color="#c0392b", label="Binomial(50, 0.2)")
ax.axvline(UCL * PADDLE, color="#c0392b", linestyle="--")
ax.axvline(LCL * PADDLE, color="#c0392b", linestyle="--")
ax.set_xlabel("Red beads in one paddle")
ax.set_ylabel("Proportion of dips")
ax.set_title("What the system produces")
ax.legend()
plt.show()

print(f"Average red beads per dip: {sim.mean():.2f}")
print(f"Share of dips inside tonight's control limits: {np.mean((sim >= LCL * PADDLE) & (sim <= UCL * PADDLE)):.1%}")
print(f"Share of dips with 3 or fewer red beads (the foreman's target): {np.mean(sim <= 3):.2%}")
if len(counts):
    worst = counts.max()
    print(f"Share of dips with {worst} or more red beads (tonight's worst result): {np.mean(sim >= worst):.1%}")

**Questions.**

- The control limits came from the formula for the standard deviation of a sample proportion. Where have you seen that formula before? (Hint: Module 5.)
- Suppose a manager looks at one worker's single bad day. How surprised should they be, given the histogram?
- Who is responsible for the red beads: the workers, or the people who designed the system and bought the raw material?

## 6. Stretch tasks (optional)

These are for anyone who finishes early, or wants to go further at home. Nothing here is on the tests.

### 6a. Manage harder

What if management fires the three worst workers every single day and hires new ones? The new hires come from the same population and use the same paddle and the same box. Run the simulation and look at the daily average.

In [ ]:
days = 30
daily_avg = []
for day in range(days):
    reds = rng.hypergeometric(RED_IN_BOX, BOX_SIZE - RED_IN_BOX, PADDLE, size=6)
    daily_avg.append(reds.mean())
    # firing the three worst and hiring three new workers changes nothing about the next day's draws

plt.figure(figsize=(10, 3.5))
plt.plot(range(1, days + 1), daily_avg, marker="o")
plt.axhline(10, color="gray", linestyle="--", label="10 red beads (20% of 50)")
plt.xlabel("Day"); plt.ylabel("Average red beads per worker")
plt.title("Thirty days of firing the three worst workers")
plt.legend(); plt.show()

Notice that the code for firing and hiring is a comment. That is the point: no line of code is needed, because who works at the paddle has no effect on what the paddle collects. Can you change something in this cell that actually lowers the daily average?

### 6b. Is the paddle really random?

Deming used the same box and paddle for years. The box held 20% red beads, so a random sample of 50 should average 10 red beads. Yet the long-run average for one of his paddles was about 9.4. One explanation is that the red beads were made differently from the white ones, so they were slightly less likely to settle into the wells.

The function `dip_paddle(red_weight=...)` lets you make red beads less likely to settle. Try a weight of 0.93, and then experiment with other values.

In [ ]:
red_weight = 0.93
dips = np.array([dip_paddle(red_weight).sum() for _ in range(2000)])

print(f"The box is {RED_IN_BOX / BOX_SIZE:.0%} red, so a random sample of 50 should average {0.2 * PADDLE:.1f} red beads.")
print(f"With red_weight = {red_weight}, the average over 2,000 dips is {dips.mean():.2f}.")

**Questions.**

- Would you notice this bias from a single dip? From tonight's 24 dips? How many dips would you need? (You may want to return to this question after our discussion of hypothesis testing)